# Example 4: Plasmasphere and Ionosphere Tutorial

This notebook demonstrates LuPNT's plasma/ionosphere module (`pylupnt.plasma`):

1. **Equatorial density slice** — sample the GCPM v2.4 electron density model on a 2-D grid
   in the magnetic equatorial plane and visualise the plasmasphere.
2. **Ray-trace from GPS to a lunar receiver** — propagate a signal from a GPS MEO satellite
   through the ionosphere/plasmasphere to an ELFO receiver and compute the TEC and dispersive delay.

**References**
- GCPM v2.4: Carpenter & Anderson (1992); Gallagher et al. (2000)
- IRI-2007: Bilitza & Reinisch (2008)
- Ray-tracing: Haselgrove (1955) / pecsim implementation


## 1. Imports and Setup


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from datetime import datetime, timezone

import pylupnt as pnt
import pylupnt.plasma as plasma

print('pylupnt plasma base path:', plasma.get_plasma_base_path())
print('RE =', plasma.RE, 'km')


[00.00][PyLuPNT] Initializing
pylupnt plasma base path: /home/kiiyama/LuPNT/data/LuPNT_data/plasma
RE = 6371.0 km


## 2. Simulation Epoch

All plasma functions accept `DateTime` (year/doy/h/m/s) for internal model queries, and
`epoch_j2000` (seconds since J2000.0) for orbit propagation and `trace_ray`.


In [2]:
YEAR, MONTH, DAY, HOUR = 2026, 1, 1, 0

mjd         = plasma.gregorian_to_mjd(YEAR, MONTH, DAY, HOUR)
epoch_j2000 = plasma.mjd_to_tj2000(mjd)   # seconds since J2000 (UTC)
dt          = plasma.mjd_to_datetime(mjd)  # DateTime for GCPM/IRI queries

# Kp index: use file-based table (returns -1 if data not found; fall back to 2)
kp = plasma.get_kp_index(dt)
if kp < 0:
    kp = 2.0
    print(f'Kp lookup failed — using default Kp = {kp}')

print(f'Epoch : {YEAR}-{MONTH:02d}-{DAY:02d} {HOUR:02d}:00 UTC')
print(f'J2000 : {epoch_j2000:.0f} s')
print(f'Kp    : {kp:.2f}')

# Initialise the IRI electron density model (used inside GCPM)
plasma.set_iri_model('IRI2007')
print('IRI model:', plasma.get_iri_model())


Epoch : 2026-01-01 00:00 UTC
J2000 : 820497627 s
Kp    : 2.00
IRI model: IRI 2007


---
## Part 1 — Equatorial Electron Density Slice

`gcpm_v24(dt, r_RE, amlt, alatr, kp)` returns `[ne_cm3, H+, He+, O+]` densities.
- `r_RE`   : geocentric distance in Earth radii
- `amlt`   : magnetic local time [0, 24 h]
- `alatr`  : magnetic latitude [deg]; 0 = equatorial plane


In [ ]:
RE_km  = plasma.RE   # 6371 km
N      = 101
R_MAX  = 8.0         # Earth radii

x_RE = np.linspace(-R_MAX, R_MAX, N)
y_RE = np.linspace(-R_MAX, R_MAX, N)
XX, YY = np.meshgrid(x_RE, y_RE)

ne_grid = np.full((N, N), np.nan)

print(f'Sampling GCPM on {N}x{N} equatorial grid (r < {R_MAX} Rₑ)...')
for i in range(N):
    for j in range(N):
        x, y = XX[i, j], YY[i, j]
        r = np.hypot(x, y)
        if r < 1.01 or r > R_MAX - 0.01:
            continue
        amlt = np.arctan2(y, x) / np.pi * 12.0 + 12.0
        if amlt > 24.0:
            amlt -= 24.0
        ne = plasma.gcpm_v24(dt, r, amlt, 0.0, kp)[0]
        if ne > 0:
            ne_grid[i, j] = ne

print('Done.')
ne_valid = ne_grid[np.isfinite(ne_grid)]
print(f'ne range: {ne_valid.min():.2e} – {ne_valid.max():.2e} cm⁻³')


Sampling GCPM on 101x101 equatorial grid (r < 8.0 Rₑ)...


In [ ]:
theta_circ = np.linspace(0, 2 * np.pi, 360)

fig, ax = plt.subplots(figsize=(8, 7))

ne_log = np.log10(ne_grid)  # NaN outside domain
im = ax.pcolormesh(XX, YY, ne_log, cmap='plasma', vmin=0, vmax=5, shading='auto')
cbar = fig.colorbar(im, ax=ax, label=r'$\log_{10}(n_e)$ [cm$^{-3}$]')

# Earth
ax.fill(np.cos(theta_circ), np.sin(theta_circ), color='steelblue', zorder=5, label='Earth')
ax.plot(np.cos(theta_circ), np.sin(theta_circ), 'k-', lw=1.5, zorder=6)

# Approximate plasmapause at 4 Rₑ
ax.plot(4 * np.cos(theta_circ), 4 * np.sin(theta_circ),
        'w--', lw=1.0, alpha=0.6, label='~plasmapause (4 Rₑ)')

# GPS MEO orbit
r_gps = 26560 / RE_km
ax.plot(r_gps * np.cos(theta_circ), r_gps * np.sin(theta_circ),
        'y-', lw=0.8, alpha=0.5, label=f'GPS orbit ({r_gps:.1f} Rₑ)')

ax.set_xlabel('X [Rₑ]', fontsize=12)
ax.set_ylabel('Y [Rₑ]', fontsize=12)
ax.set_title(
    f'Equatorial Electron Density — GCPM v2.4 + IRI-2007\n'
    f'{YEAR}-{MONTH:02d}-{DAY:02d} {HOUR:02d}:00 UTC,  Kp = {kp:.1f}',
    fontsize=11
)
ax.set_xlim(-R_MAX, R_MAX)
ax.set_ylim(-R_MAX, R_MAX)
ax.set_aspect('equal')
ax.legend(loc='upper right', fontsize=8)
ax.grid(alpha=0.15)
plt.tight_layout()
plt.show()


---
## Part 2 — Ray-Trace from GPS to a Lunar (ELFO) Receiver

We propagate a single GPS satellite and an ELFO receiver from their orbital elements, then
run `trace_ray` to compute the integrated TEC and L1 dispersive delay along the link.

**Orbits** (same as Example 5)
| Satellite | a [km] | e | i [deg] |
|-----------|--------|---|--------|
| GPS MEO   | 26 560 | 0.001 | 55 |
| ELFO      | 6 541.4 | 0.6 | 56.2 |


In [ ]:
# ── Orbital elements ────────────────────────────────────────────────────────
DEG = np.pi / 180

# GPS satellite in circular MEO (a, e, i, RAAN, omega, M) — all angles in rad
coe_gps  = np.array([26560.0, 0.001, 55.0*DEG, 0.0, 0.0, 0.0])
gps_sat  = plasma.Satellite(1, coe_gps, epoch_j2000, plasma.GM_EARTH)
pos_tx_km = np.array(gps_sat.get_pos())   # ECI [km]

# ELFO receiver: a=6541.4 km, e=0.6, i=56.2°, ω=90° (frozen)
# Use M=π (apoapsis) so r = a(1+e) = 6541.4*1.6 ≈ 10466 km (above Earth's surface)
coe_elfo = np.array([6541.4, 0.6, 56.2*DEG, 0.0, 90.0*DEG, np.pi])
elfo_sat = plasma.Satellite(100, coe_elfo, epoch_j2000, plasma.GM_EARTH)
pos_rx_km = np.array(elfo_sat.get_pos())  # ECI [km]

r_tx = np.linalg.norm(pos_tx_km)
r_rx = np.linalg.norm(pos_rx_km)
print(f'GPS  position : {pos_tx_km}  km')
print(f'             |r| = {r_tx:.0f} km = {r_tx/RE_km:.2f} Rₑ')
print(f'ELFO position : {pos_rx_km}  km')
print(f'             |r| = {r_rx:.0f} km = {r_rx/RE_km:.2f} Rₑ')

In [ ]:
# ── RayTraceConfig ──────────────────────────────────────────────────────────
# Cutoff: only model plasma below 4 Rₑ; above that the medium is vacuum.
CUTOFF_RE = 4.0

config = plasma.RayTraceConfig()
config.freq_Hz            = plasma.freq_L1   # 1575.42 MHz
config.step_size          = 100.0            # integration step [km]
config.correction         = True             # apply receiver-position correction
config.fine_correction    = False            # skip iterative fine correction
config.correction_method  = 'neldermead'         # solver for position correction
config.cutoff_r           = CUTOFF_RE * RE_km
config.kp                 = kp
config.gradn_dx           = 1.0             # refractive-index gradient step [km]
config.integ_method       = 'rk4'
config.use_fortran_gcpm   = False
config.straight_ray       = False

print(f'Frequency   : {config.freq_Hz*1e-6:.3f} MHz  (L1)')
print(f'Step size   : {config.step_size:.0f} km')
print(f'Cutoff      : {CUTOFF_RE} Rₑ = {config.cutoff_r:.0f} km')
print(f'Kp          : {config.kp:.2f}')

In [ ]:
print('Running ray-trace (correction=True, fine_correction=False) ...')
pp = plasma.trace_ray(epoch_j2000, pos_tx_km, pos_rx_km, config,
                      debug_prop=False, debug_corr=False)

print()
print('=== Ray-Trace Results ===')
print(f'  TEC (total)      : {pp.tecu:.3f} TECU')
print(f'  L1 total delay   : {pp.total_delay_m:.4f} m')
print(f'    TEC delay      : {pp.tec_delay_m:.4f} m')
print(f'    Bending delay  : {pp.dist_bend_m:.6f} m')
print(f'  Straight dist    : {pp.dist_straight_km:.1f} km')


In [ ]:
# ── Visualise ray path and density profile ───────────────────────────────────
pos_path = np.array(pp.pos_eci)   # [N_steps x 3] km, ECI
r_path   = np.array(pp.r)         # geocentric distance [km]
s_path   = np.array(pp.s)         # arc length [km]
tec_sec  = np.array(pp.tec_section)  # TECU per segment

alt_path = r_path - RE_km         # altitude [km]

# Electron density along path (re-compute cheaply at each step)
ne_path = np.array([
    plasma.compute_ne(epoch_j2000, pos_path[k], config)
    for k in range(len(s_path))
])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- Plot A: ray path in XZ plane ------------------------------------------
ax = axes[0]
sc = ax.scatter(pos_path[:, 0] / RE_km, pos_path[:, 2] / RE_km,
                c=np.log10(np.where(ne_path > 0, ne_path, np.nan)),
                cmap='plasma', s=6, vmin=0, vmax=5)
fig.colorbar(sc, ax=ax, label=r'$\log_{10}(n_e)$ [cm$^{-3}$]', shrink=0.8)
ax.scatter(*pos_tx_km[[0, 2]] / RE_km, marker='*', s=250, color='gold',
           zorder=10, label='GPS Tx')
ax.scatter(*pos_rx_km[[0, 2]] / RE_km, marker='^', s=120, color='cyan',
           zorder=10, label='ELFO Rx')
# Earth
ax.fill(np.cos(theta_circ), np.sin(theta_circ), color='steelblue', zorder=5)
ax.plot(np.cos(theta_circ), np.sin(theta_circ), 'k-', lw=1.5, zorder=6)
ax.plot(CUTOFF_RE * np.cos(theta_circ), CUTOFF_RE * np.sin(theta_circ),
        'w--', lw=0.8, alpha=0.5, label=f'cutoff ({CUTOFF_RE} Rₑ)')
ax.set_xlabel('X [Rₑ]'); ax.set_ylabel('Z [Rₑ]')
ax.set_title('Ray Path (XZ plane)\n(coloured by $n_e$)')
ax.legend(fontsize=8)
ax.set_aspect('equal')
ax.grid(alpha=0.2)

# --- Plot B: altitude profile -----------------------------------------------
ax2 = axes[1]
ax2.plot(s_path, alt_path, 'b-', lw=1.5)
ax2.axhline(0, color='k', lw=0.5, ls='--')
ax2.axhline((CUTOFF_RE - 1) * RE_km, color='gray', lw=0.8, ls=':',
            label=f'cutoff {CUTOFF_RE} Rₑ')
ax2.set_xlabel('Arc length along ray [km]')
ax2.set_ylabel('Altitude [km]')
ax2.set_title('Altitude Profile Along Ray')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# --- Plot C: electron density profile ---------------------------------------
ax3 = axes[2]
ne_plot = np.where(ne_path > 0, ne_path, np.nan)
ax3.semilogy(s_path, ne_plot, 'r-', lw=1.5)
ax3.set_xlabel('Arc length along ray [km]')
ax3.set_ylabel(r'$n_e$ [cm$^{-3}$]')
ax3.set_title(r'Electron Density Along Ray')
ax3.grid(alpha=0.3, which='both')

fig.suptitle(
    f'GPS → ELFO Ray-Trace  |  L1  |  Kp={kp:.1f}  |  '
    f'TEC = {pp.tecu:.2f} TECU  |  L1 delay = {pp.total_delay_m:.3f} m',
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()
